# 1. Simulation Horizon & Demand-Volume Calibration

Calibrates the two simulation knobs that set both fidelity and per-evaluation runtime, using **one persistent worker pool** (workers + cached `TravelGraph` reused across the entire sweep — no ~90s reload or TravelGraph rebuild per evaluation).

- **Δt is fixed at 10s.** The temporal-discretization sweep is retired: sub-tick rounding at stops is unbiased and averages out over the stochastic passenger population.
- **Phase 1 — Horizon sweep:** passenger completion rate vs `num_ticks` at a fixed moderate density. Pick the horizon where completion plateaus.
- **Phase 2 — Volume sweep:** fitness CV across replications vs spawn volume at a fixed moderate horizon. Pick the volume just past the variance collapse (CV ≤ 5%).

The two sweeps deliberately never combine max-ticks × max-density (the expensive corner).

**Outputs:** `rnd/csv/rnd1_horizon.csv`, `rnd/csv/rnd1_volume.csv`, `rnd/images/rnd1_horizon_and_volume.png`

In [1]:
import os, sys, time, gc, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.getcwd())
random.seed(42)
np.random.seed(42)

from utils_simplified import reuse_citygraph, reuse_ddm, generate_route_system, run_reps_overrides

cg_pkl  = "rnd/pkl/profile_p1.pkl"
ddm_pkl = "rnd/pkl/ddm_8am.pkl"

print("Loading cached CityGraph and DDM...")
t0 = time.time()
G_city = reuse_citygraph(cg_pkl)
ddm    = reuse_ddm(ddm_pkl)
print(f"Loaded in {time.time()-t0:.1f}s | {len(G_city.nodes)} nodes, {len(G_city.graph)} edges")

Loading cached CityGraph and DDM...
[INFO] Reusing CityGraph from pickle file: rnd/pkl/profile_p1.pkl
[INFO] Reusing DirectDemandSampler from pickle file: rnd/pkl/ddm_8am.pkl
Loaded in 5.9s | 36866 nodes, 76310 edges


In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
ROUTE_COUNTS     = [5, 10]                       # production (5) + a denser stress case; add more if time allows
PAX_PER_ROUTE_HR = [50, 150, 300]                # off-peak / standard / rush — for the VOLUME (CV) sweep
HORIZON_TICKS    = [120, 240, 360, 480, 600, 720] # for the COMPLETION-vs-horizon sweep
REPLICATIONS     = 7                             # stochastic reps per point (CV needs >= 2)
JEEPS_PER_ROUTE  = 10
SPT              = 10                             # Δt fixed at 10s (discretization sweep retired)
CV_THRESHOLD     = 0.05

# Anchors chosen so we NEVER hit the expensive max-ticks x max-density corner:
HORIZON_DENSITY  = 50   # pax/route/hr held fixed while sweeping the horizon
VOLUME_TICKS     = 360   # num_ticks held fixed while sweeping volume

os.makedirs("rnd/csv", exist_ok=True)
os.makedirs("rnd/images", exist_ok=True)

# ── Open ONE persistent worker pool ──────────────────────────────────────────
# Workers load CityGraph + DDM ONCE and cache the TravelGraph per route set, so
# every sweep point below reuses them (the fix for the ~90s/eval + TG-rebuild cost).
import yaml
from utils.simulation_parallel import ParallelSimulationRunner

with open("configs/profile_p1.yaml", "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)
# Inject pickle paths so workers LOAD the graphs (fast) instead of rebuilding from PBF (slow).
base_cfg["cg_pkl"]  = cg_pkl
base_cfg["ddm_pkl"] = ddm_pkl

MAX_WORKERS = min(REPLICATIONS, max(1, (os.cpu_count() or 2) - 1))

# Guard against leaving a stale pool open if this cell is re-run.
try:
    runner.close_pool()
except Exception:
    pass

runner = ParallelSimulationRunner(config=base_cfg, max_workers=MAX_WORKERS)
runner.open_pool()
print(f"Persistent pool open with {MAX_WORKERS} workers.")
print(f"Routes: {ROUTE_COUNTS} | densities: {PAX_PER_ROUTE_HR} | horizons: {HORIZON_TICKS} | SPT={SPT}s")

[ParallelRunner] Opened persistent pool with 3 workers.
Persistent pool open with 3 workers.
Routes: [5, 10] | densities: [50, 150, 300] | horizons: [120, 240, 360, 480, 600, 720] | SPT=10s


In [3]:
# ==========================================
# FUNCTIONS
# ==========================================

def compute_cv(scores):
    """Coefficient of variation of fitness across replications."""
    if len(scores) < 2:
        return float('nan')
    arr = np.array(scores, dtype=float)
    m = arr.mean()
    return float(arr.std(ddof=1) / m) if abs(m) > 1e-9 else float('nan')

def eval_point(routes, num_ticks, spawn_rate, n_jeeps):
    """
    Run REPLICATIONS stochastic reps of `routes` at one (num_ticks, spawn_rate) point
    through the persistent pool. Returns CV, mean fitness, and mean completion fraction.
    Heavy objects are reused across calls — only these scalars change per call.
    """
    overrides = {
        "seconds_per_tick":        SPT,
        "num_ticks":               int(num_ticks),
        "spawn_rate_per_hour":     float(spawn_rate),
        "total_allocatable_jeeps": int(n_jeeps),
    }
    results = run_reps_overrides(runner, routes, REPLICATIONS, overrides)

    scores, comps = [], []
    for r in results:
        if r is None:
            continue
        scores.append(r.score)
        c   = r.metrics.get("completed_count", 0)
        inc = r.metrics.get("incomplete_count", 0)
        tot = c + inc
        comps.append(c / tot if tot > 0 else np.nan)

    return {
        "cv":              compute_cv(scores),
        "mean_fit":        float(np.mean(scores)) if scores else float('nan'),
        "mean_completion": float(np.nanmean(comps)) if comps else float('nan'),
        "n_ok":            len(scores),
    }

In [ ]:
# ==========================================
# MAIN EXECUTION — two cheap 1D sweeps
# ==========================================
horizon_rows, volume_rows = [], []

for n_routes in ROUTE_COUNTS:
    random.seed(42 + n_routes)
    np.random.seed(42 + n_routes)
    print(f"\n{'='*70}\nGENERATING {n_routes} ROUTES...")
    t0 = time.time()
    routes  = generate_route_system(n_routes, G_city, ddm)
    n_jeeps = n_routes * JEEPS_PER_ROUTE
    print(f"  {n_routes} routes, {sum(len(r.path) for r in routes)} edges, {n_jeeps} jeeps | {time.time()-t0:.1f}s")
    print(f"  (first eval per route set builds + caches the TravelGraph in each worker; later evals reuse it)")

    # ── Phase 1: Horizon sweep (fixed moderate density) ──
    rate = HORIZON_DENSITY * n_routes
    print(f"\n  [Horizon sweep] density={HORIZON_DENSITY}/route ({rate}/hr)")
    for T in HORIZON_TICKS:
        t0 = time.time()
        res = eval_point(routes, T, rate, n_jeeps)
        print(f"    ticks={T:4d} ({T*SPT/60:4.0f} min) | completion={res['mean_completion']*100:5.1f}% "
              f"| CV={res['cv']:.4f} | mean={res['mean_fit']:.0f} | {time.time()-t0:.0f}s")
        horizon_rows.append({"num_routes": n_routes, "density": HORIZON_DENSITY, "total_rate": rate,
                             "num_ticks": T, "sim_min": T*SPT/60,
                             "mean_completion": res['mean_completion'], "cv": res['cv'], "mean_fit": res['mean_fit']})

    # ── Phase 2: Volume sweep (fixed moderate horizon) ──
    print(f"\n  [Volume sweep] ticks={VOLUME_TICKS} ({VOLUME_TICKS*SPT/60:.0f} min)")
    for density in PAX_PER_ROUTE_HR:
        rate = density * n_routes
        t0 = time.time()
        res = eval_point(routes, VOLUME_TICKS, rate, n_jeeps)
        status = "PASS" if (res['cv'] <= CV_THRESHOLD) else "FAIL"
        print(f"    density={density:3d}/route ({rate:5.0f}/hr) | CV={res['cv']:.4f} [{status}] "
              f"| completion={res['mean_completion']*100:5.1f}% | {time.time()-t0:.0f}s")
        volume_rows.append({"num_routes": n_routes, "density": density, "total_rate": rate,
                            "num_ticks": VOLUME_TICKS, "cv": res['cv'],
                            "mean_completion": res['mean_completion'], "mean_fit": res['mean_fit']})

    # incremental save so a crash never loses completed work
    pd.DataFrame(horizon_rows).to_csv("rnd/csv/rnd1_horizon.csv", index=False)
    pd.DataFrame(volume_rows).to_csv("rnd/csv/rnd1_volume.csv", index=False)

    del routes
    gc.collect()

print(f"\n{'='*70}\nEXPERIMENT COMPLETE — rnd/csv/rnd1_horizon.csv + rnd/csv/rnd1_volume.csv\n{'='*70}")


GENERATING 5 ROUTES...
[INFO] Generating 5 routes...
  5 routes, 17442 edges, 50 jeeps | 1.0s
  (first eval per route set builds + caches the TravelGraph in each worker; later evals reuse it)

  [Horizon sweep] density=150/route (750/hr)
[ParallelRunner] Sweeping 7 setups across 3 cores (override mode)...


In [ ]:
# ==========================================
# VISUALIZE
# ==========================================
hdf = pd.read_csv("rnd/csv/rnd1_horizon.csv")
vdf = pd.read_csv("rnd/csv/rnd1_volume.csv")

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Completion vs simulation horizon → pick the plateau as production num_ticks
for nr, grp in hdf.groupby("num_routes"):
    grp = grp.sort_values("sim_min")
    axes[0].plot(grp["sim_min"], grp["mean_completion"] * 100, marker="o", label=f"{nr} routes")
axes[0].axhline(95, color="red", linestyle="--", label="95% completion")
axes[0].set_xlabel("Simulated Horizon (min)")
axes[0].set_ylabel("Passenger Completion (%)")
axes[0].set_title("(a) Completion vs Simulation Horizon")
axes[0].legend(title="Route count")

# (b) Fitness CV vs spawn volume → pick the volume past the collapse (CV ≤ 0.05)
for nr, grp in vdf.groupby("num_routes"):
    grp = grp.sort_values("total_rate")
    axes[1].plot(grp["total_rate"], grp["cv"], marker="o", label=f"{nr} routes")
axes[1].axhline(0.05, color="red", linestyle="--", label="5% CV threshold")
axes[1].set_xlabel("Total Spawn Rate (pax/hr)")
axes[1].set_ylabel("Fitness CV across replications")
axes[1].set_title("(b) Fitness Variance Collapse vs Volume")
axes[1].legend(title="Route count")

fig.suptitle("Simulation Horizon & Demand-Volume Calibration", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("rnd/images/rnd1_horizon_and_volume.png", dpi=150, bbox_inches="tight")
plt.show()

# Close the persistent pool now that the sweep is done (frees the worker processes).
try:
    runner.close_pool()
    print("Persistent pool closed.")
except Exception as e:
    print(f"(pool already closed: {e})")